<a href="https://colab.research.google.com/github/nguyenduydoanh/H-C-M-Y/blob/main/Ki%E1%BB%83m_Tra_H%E1%BB%8Dc_M%C3%A1y_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PHẦN 1 : KẾT NỐI GOOGLE DRIVE & LOAD DỮ LIỆU

In [3]:
import pandas as pd
import os

# ==========================================
# 1. KẾT NỐI VỚI GOOGLE DRIVE TRÊN COLAB
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Định nghĩa đường dẫn tới thư mục 'kiemtrahocmay' của bạn trên Google Drive
folder_path = '/content/drive/MyDrive/Học Máy/'

# ==========================================
# 2. LOAD CÁC FILE CSV VÀ EXCEL TỪ THƯ MỤC DRIVE
# ==========================================
customer = pd.read_csv(os.path.join(folder_path, "1.Data_Customer.csv"))
transaction = pd.read_csv(os.path.join(folder_path, "2.Data_MyVIB_Transaction.csv"))
activity = pd.read_csv(os.path.join(folder_path, "3.Data_MyVIB_Activity.csv"))
deposit = pd.read_csv(os.path.join(folder_path, "4.Data_Deposit.csv"))

lending = pd.read_excel(os.path.join(folder_path, "5.Data_Lending.xlsx"))
card = pd.read_excel(os.path.join(folder_path, "6.Data_Card.xlsx"))

# ==========================================
# 3. CHECK SHAPE (Kiểm tra kích thước các bảng)
# ==========================================
print("Customer:", customer.shape)
print("Transaction:", transaction.shape)
print("Activity:", activity.shape)
print("Deposit:", deposit.shape)
print("Lending:", lending.shape)
print("Card:", card.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_1366/1477704341.py:16: DtypeWarning: Columns (5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  customer = pd.read_csv(os.path.join(folder_path, "1.Data_Customer.csv"))
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Customer: (290223, 9)
Transaction: (1418030, 8)
Activity: (16132675, 6)
Deposit: (1258424, 6)
Lending: (576431, 4)
Card: (871589, 3)


PHẦN 2: XỬ LÝ FILE TARGET (TỪ FILE CARD)

In [4]:
import pandas as pd
import numpy as np
import os

folder_path = '/content/drive/MyDrive/Học Máy/'

# Load file Card
card = pd.read_excel(os.path.join(folder_path, "6.Data_Card.xlsx"))
card.columns = card.columns.str.strip().str.upper()

# Chọn cột cần dùng
target = card[["CUSTOMER_NUMBER", "COUNT_CREDITCARD", "MONTH"]].copy()
target["COUNT_CREDITCARD"] = pd.to_numeric(target["COUNT_CREDITCARD"], errors="coerce").fillna(0)
target["MONTH"] = pd.to_datetime(target["MONTH"], errors="coerce")

# Gom theo khách hàng
summary = target.groupby("CUSTOMER_NUMBER").agg(
    MAX_CARD=("COUNT_CREDITCARD", "max"),
    MIN_MONTH=("MONTH", "min"),
    MAX_MONTH=("MONTH", "max")
).reset_index()

summary["TARGET"] = (summary["MAX_CARD"] > 0).astype(int)
summary["TARGET_MONTH"] = np.where(
    summary["TARGET"] == 1,
    summary["MIN_MONTH"],
    summary["MAX_MONTH"]
)

target_final = summary[["CUSTOMER_NUMBER", "TARGET", "TARGET_MONTH"]]
target_final.to_csv(os.path.join(folder_path, "target_processed.csv"), index=False)

print(target_final.head())
print("Saved: target_processed.csv thành công!")

   CUSTOMER_NUMBER  TARGET TARGET_MONTH
0                0       0   2019-12-31
1                3       0   2019-12-31
2                8       1   2019-05-31
3                9       0   2019-12-31
4               13       0   2019-12-31
Saved: target_processed.csv thành công!


PHẦN 3: NỐI CUSTOMER VỚI TARGET & LỌC NGÀY MỞ TÀI KHOẢN

In [5]:
import pandas as pd
import os

folder_path = '/content/drive/MyDrive/Học Máy/'

customer = pd.read_csv(os.path.join(folder_path, "1.Data_Customer.csv"))
target = pd.read_csv(os.path.join(folder_path, "target_processed.csv"))

customer.columns = customer.columns.str.strip().str.upper()
target.columns = target.columns.str.strip().str.upper()

customer["CLIENT_CREATE_DATE"] = pd.to_datetime(customer["CLIENT_CREATE_DATE"], errors="coerce")
target["TARGET_MONTH"] = pd.to_datetime(target["TARGET_MONTH"], errors="coerce")

# Inner join
df = customer.merge(target, on="CUSTOMER_NUMBER", how="inner")
print("Before filtering:", df.shape)

# Lọc bỏ ngày mở thẻ trước ngày mở tài khoản
df = df[df["TARGET_MONTH"] >= df["CLIENT_CREATE_DATE"]]
print("After filtering:", df.shape)

df.to_csv(os.path.join(folder_path, "step2_customer_target_filtered.csv"), index=False)
print("Saved: step2_customer_target_filtered.csv thành công!")

/tmp/ipykernel_1366/2659256474.py:6: DtypeWarning: Columns (5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  customer = pd.read_csv(os.path.join(folder_path, "1.Data_Customer.csv"))


Before filtering: (150459, 11)
After filtering: (150408, 11)
Saved: step2_customer_target_filtered.csv thành công!


PHẦN 4: NỐI TRANSACTION VÀ LỌC RÒ RỈ DỮ LIỆU (DATA LEAKAGE)

In [6]:
import pandas as pd
import os

folder_path = '/content/drive/MyDrive/Học Máy/'

target = pd.read_csv(os.path.join(folder_path, "target_processed.csv"))
trans = pd.read_csv(os.path.join(folder_path, "2.Data_MyVIB_Transaction.csv"))

target.columns = target.columns.str.strip().str.upper()
trans.columns = trans.columns.str.strip().str.upper()

target["TARGET_MONTH"] = pd.to_datetime(target["TARGET_MONTH"], errors="coerce")
trans["TRANS_DATE"] = pd.to_datetime(trans["TRANS_DATE"], errors="coerce")

# Left join giữ toàn bộ khách hàng trong target
df = target.merge(trans, on="CUSTOMER_NUMBER", how="left")
print("Before filtering:", df.shape)

# Giữ khách không có giao dịch hoặc giao dịch trước ngày mở thẻ
df = df[
    df["TRANS_DATE"].isna() |
    (
        df["TARGET_MONTH"].notna() &
        (df["TRANS_DATE"] < df["TARGET_MONTH"])
    )
]
print("After filtering:", df.shape)

df.to_csv(os.path.join(folder_path, "step3_transaction_before_card_keep_all_customers.csv"), index=False)
print("Saved: step3_transaction_before_card_keep_all_customers.csv thành công!")

Before filtering: (1404891, 10)
After filtering: (1171302, 10)
Saved: step3_transaction_before_card_keep_all_customers.csv thành công!


PHẦN 5: TẠO FEATURE CHO TRANSACTION

In [7]:
import pandas as pd
import os

folder_path = '/content/drive/MyDrive/Học Máy/'

df = pd.read_csv(os.path.join(folder_path, "step3_transaction_before_card_keep_all_customers.csv"))
df.columns = df.columns.str.strip().str.upper()

df["TRANS_AMOUNT"] = pd.to_numeric(df["TRANS_AMOUNT"], errors="coerce").fillna(0)
df["TRANS_LV1"] = df["TRANS_LV1"].astype(str).str.strip().str.upper()
df["TRANS_LV2"] = df["TRANS_LV2"].astype(str).str.strip().str.upper()

def get_mode_value(series):
    s = series.dropna().astype(str).str.strip()
    s = s[~s.isin(['NAN', 'NAT', 'NONE', ''])]
    if s.empty:
        return None
    return s.value_counts().idxmax()

feature = df.groupby("CUSTOMER_NUMBER").agg(
    TRANS_LV1_MODE=("TRANS_LV1", get_mode_value),
    TRANS_LV2_MODE=("TRANS_LV2", get_mode_value),
    TRANS_AMOUNT_MAX=("TRANS_AMOUNT", "max"),
    TRANS_AMOUNT_MEAN=("TRANS_AMOUNT", "mean"),
    TARGET=("TARGET", "max")
).reset_index()

# Encode category -> code
feature["TRANS_LV1_MODE_CODE"] = feature["TRANS_LV1_MODE"].astype("category").cat.codes
feature["TRANS_LV2_MODE_CODE"] = feature["TRANS_LV2_MODE"].astype("category").cat.codes

feature.to_csv(os.path.join(folder_path, "feature_transaction_mode.csv"), index=False)

print("Saved: feature_transaction_mode.csv thành công!")
print(feature.head())

Saved: feature_transaction_mode.csv thành công!
   CUSTOMER_NUMBER TRANS_LV1_MODE TRANS_LV2_MODE  TRANS_AMOUNT_MAX  \
0                0       TRANSFER    OUTSIDE_VIB        99800000.0   
1                3           None           None               0.0   
2                8           None           None               0.0   
3                9          TOPUP         MOBILE         5000000.0   
4               13           None           None               0.0   

   TRANS_AMOUNT_MEAN  TARGET  TRANS_LV1_MODE_CODE  TRANS_LV2_MODE_CODE  
0       2.498190e+07       0                    2                    9  
1       0.000000e+00       0                   -1                   -1  
2       0.000000e+00       1                   -1                   -1  
3       1.949163e+06       0                    1                    8  
4       0.000000e+00       0                   -1                   -1  
